# Usando la API de Gemini con Python
 
En el notebook anterior exploramos modelos open-source con Hugging Face,
tanto local como remotamente.
 
En este notebook usaremos **Gemini 2.5 Flash** de Google, uno de los modelos
más capaces disponibles hoy con un generoso free tier.

![Alt text](figs/01/gemini_api_architecture.svg) 
 
## Por qué Gemini 2.5 Flash?
 
- **Gratis**: 500 requests/día, 1M tokens/minuto
- **Capaz**: comparable a GPT-4 en muchas tareas
- **Rápido**: optimizado para latencia baja
- **Multimodal**: acepta texto, imágenes, audio y video (lo exploraremos más adelante)
 
## Prerrequisitos
 
### Obteniendo tu API Key de Gemini
 
1. Ve a [aistudio.google.com](https://aistudio.google.com)
2. Inicia sesión con tu cuenta de Google
3. Click en "Get API Key" en el panel izquierdo
4. Click en "Create API Key"
5. Copia la key generada
6. Crea un archivo `.env` en la raíz de tu proyecto:
 
```
GEMINI_API_KEY="tu_key_aqui"
```
 
### Límites del free tier
- 500 requests por día
- 1,000,000 tokens por minuto
- Suficiente para el curso completo
 
IMPORTANTE: Nunca pongas la API key directamente en el código.
Siempre cárgala desde variables de entorno.
 
### Instalación
"""

In [ ]:
%%bash
pip install google-genai python-dotenv

## 1. Configuración
 
El SDK de Gemini es `google-genai`. La inicialización es simple:
un cliente con tu API key y listo.

In [3]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [ ]:
# Cargamos las variables de entorno desde el archivo .env
load_dotenv()

In [ ]:
# Verificamos que la key esté disponible
if os.getenv("GEMINI_API_KEY"):
    print("Gemini API Key cargada correctamente")
else:
    print("ERROR: GEMINI_API_KEY no encontrada. Verifica tu archivo .env")

In [5]:
# Inicializamos el cliente de Gemini con nuestra API key
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [11]:
# Nombre del modelo que usaremos en todo el notebook
MODELO = "gemini-2.5-flash-lite"

## 2. Primera Llamada a la API
 
Enviamos un mensaje simple al modelo y obtenemos una respuesta.
Nota que la estructura es muy similar a OpenAI y a HuggingFace:
siempre hay un mensaje de sistema y un mensaje del usuario.

In [ ]:
# Llamada básica al modelo
response = client.models.generate_content(
    model=MODELO,
    config=types.GenerateContentConfig(
        system_instruction="Eres un asistente util.",
    ),
    contents="Escribe una historia de una oración sobre un unicornio."
)
 
print(response.text)

## 3. Entendiendo la Estructura de la Respuesta
 
La respuesta contiene metadatos útiles además del texto generado.
Veamos qué información está disponible:

In [ ]:
response = client.models.generate_content(
    model=MODELO,
    config=types.GenerateContentConfig(
        system_instruction="Eres un asistente util.",
    ),
    contents="Hola, como estas?"
)
 
# El texto de la respuesta
print("Respuesta:")
print(response.text)
 
# Metadatos de uso de tokens
print("\nUso de tokens:")
print(f"  Tokens de entrada: {response.usage_metadata.prompt_token_count}")
print(f"  Tokens de salida: {response.usage_metadata.candidates_token_count}")
print(f"  Total de tokens: {response.usage_metadata.total_token_count}")

Los tokens son importantes porque determinan el costo de uso.
En el free tier no pagas, pero en producción cada token tiene un precio.
Monitorear el uso de tokens es una buena práctica desde el inicio.
 
## 4. Interfaz de Chat Sin Memoria
 
Construyamos una función de chat simple, igual que en los notebooks anteriores.
Cada llamada es independiente — el modelo no recuerda interacciones previas.
 
### Definir la función

In [14]:
def get_response_sin_memoria(user_message: str) -> str:
    """Obtiene una respuesta de Gemini sin historial de conversacion."""
    response = client.models.generate_content(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction="Eres un asistente util.",
        ),
        contents=user_message
    )
    return response.text

### Primera interacción

In [ ]:
# Primera pregunta
pregunta = "Que es la inteligencia artificial?"
respuesta = get_response_sin_memoria(pregunta)
 
print(f"Tu: {pregunta}")
print(f"Asistente: {respuesta}")

### Pregunta de seguimiento (sin memoria)

In [ ]:
# Segunda pregunta — el modelo no recuerda la anterior
pregunta = "Puedes elaborar mas sobre eso?"
respuesta = get_response_sin_memoria(pregunta)
 
print(f"Tu: {pregunta}")
print(f"Asistente: {respuesta}")

Como puedes ver, el modelo no sabe a qué se refiere "eso" porque no tiene
contexto de la conversación anterior. Igual que vimos con Ollama y HuggingFace.
 
## 5. Interfaz de Chat Con Memoria
 
Ahora construimos un chat que mantiene el historial de conversación.
Gemini usa objetos `Content` para representar los mensajes del historial.

### Configurar la memoria

In [17]:
# El historial de conversacion es una lista de objetos Content
# Cada Content tiene un role ("user" o "model") y una lista de partes
conversation_memory = []
 
def chat_con_memoria(user_message: str) -> str:
    """Chat con Gemini manteniendo el historial de conversacion."""
 
    # Agregamos el mensaje del usuario al historial
    conversation_memory.append(
        types.Content(
            role="user",
            parts=[types.Part(text=user_message)]
        )
    )
 
    # Llamamos al modelo con todo el historial
    response = client.models.generate_content(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction="Eres un asistente util.",
        ),
        contents=conversation_memory  # pasamos el historial completo
    )
 
    assistant_response = response.text
 
    # Agregamos la respuesta del modelo al historial
    conversation_memory.append(
        types.Content(
            role="model",  # en Gemini el rol del asistente es "model", no "assistant"
            parts=[types.Part(text=assistant_response)]
        )
    )
 
    # Mostramos el intercambio y el uso de tokens
    print(f"Tu: {user_message}")
    print(f"Asistente: {assistant_response}")
    print(f"[Tokens usados: {response.usage_metadata.total_token_count}]")
    print(f"[Mensajes en historial: {len(conversation_memory)}]")
 
    return assistant_response

### Primera interacción con memoria

In [ ]:
# Primera pregunta
chat_con_memoria("Que es la inteligencia artificial?")

### Pregunta de seguimiento (con memoria)

In [ ]:
# Segunda pregunta — ahora sí recuerda el contexto
chat_con_memoria("Puedes elaborar mas sobre eso?")

### Ver el historial completo

In [ ]:
print("Historial de conversacion:")
print("-" * 40)
for mensaje in conversation_memory:
    rol = "Tu" if mensaje.role == "user" else "Asistente"
    print(f"{rol}: {mensaje.parts[0].text}\n")

## 6. Streaming de Respuestas
 
El streaming muestra la respuesta token por token a medida que se genera,
igual que la experiencia en Gemini.ai o ChatGPT.

In [22]:
def stream_response(user_message: str) -> str:
    """Obtiene una respuesta de Gemini en modo streaming."""
 
    print(f"Tu: {user_message}")
    print("Asistente: ", end="", flush=True)
 
    respuesta_completa = ""
 
    # generate_content_stream devuelve un iterador de chunks
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction="Eres un asistente util.",
        ),
        contents=user_message
    ):
        if chunk.text:
            respuesta_completa += chunk.text
            print(chunk.text, end="", flush=True)
 
    print("\n")
    return respuesta_completa

In [ ]:
# Probamos el streaming
stream_response("Escribe un poema corto sobre la programacion.")

## 7. Streaming con Memoria
 
Combinamos streaming con historial de conversación:

In [24]:
streaming_conversation = []
 
def stream_chat_con_memoria(user_message: str) -> str:
    """Chat con memoria y streaming."""
 
    # Agregamos el mensaje del usuario al historial
    streaming_conversation.append(
        types.Content(
            role="user",
            parts=[types.Part(text=user_message)]
        )
    )
 
    print(f"Tu: {user_message}")
    print("Asistente: ", end="", flush=True)
 
    respuesta_completa = ""
 
    # Streaming con historial completo
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction="Eres un asistente util.",
        ),
        contents=streaming_conversation
    ):
        if chunk.text:
            respuesta_completa += chunk.text
            print(chunk.text, end="", flush=True)
 
    print("\n")
 
    # Agregamos la respuesta al historial
    streaming_conversation.append(
        types.Content(
            role="model",
            parts=[types.Part(text=respuesta_completa)]
        )
    )
 
    return respuesta_completa

In [ ]:
# Primera pregunta con streaming y memoria
stream_chat_con_memoria("Cuales son las tres leyes de la robotica?")

In [ ]:
# Pregunta de seguimiento
stream_chat_con_memoria("Quien creo esas leyes?")
 

## 8. Entendiendo los Roles en Gemini
 
A diferencia de OpenAI donde el asistente usa el rol "assistant",
en Gemini el rol es "model". Los roles disponibles son:

In [ ]:
# Ejemplo con historial manual que muestra los roles
response = client.models.generate_content(
    model=MODELO,
    config=types.GenerateContentConfig(
        system_instruction="Eres un pirata que solo habla en jerga pirata.",
    ),
    contents=[
        types.Content(role="user", parts=[types.Part(text="Hola, como estas?")]),
        types.Content(role="model", parts=[types.Part(text="Arrr! Estoy de lo mas bien, marinero!")]),
        types.Content(role="user", parts=[types.Part(text="Cuentame sobre el clima.")]),
    ]
)
 
print(response.text)

## 9. Ventana de Contexto de Gemini
 
Gemini 2.5 Flash tiene una ventana de contexto de **1,000,000 tokens** —
significativamente mayor que la mayoría de modelos.
 
Para referencia:
- **TinyLlama**: ~2,048 tokens
- **GPT-4**: hasta 128,000 tokens
- **Gemini 2.5 Flash**: 1,000,000 tokens (~750,000 palabras)
 
Esto reduce considerablemente el problema del olvido en conversaciones largas,
aunque no lo elimina completamente. Las mismas estrategias del notebook anterior
(ventana deslizante, resumen) siguen siendo relevantes para casos extremos.
 
## 10. Gestionando Conversaciones Largas

Las mismas estrategias que vimos antes aplican aquí:

In [28]:
def trim_conversation(messages: list, max_messages: int = 10) -> list:
    """Conserva solo los N mensajes mas recientes mas el contexto del sistema."""
    if len(messages) > max_messages:
        return messages[-max_messages:]
    return messages
 
def summarize_conversation(messages: list) -> list:
    """Resume la conversacion para reducir el uso de tokens."""
 
    # Construimos un texto con el historial para resumir
    historial_texto = "\n".join([
        f"{'Usuario' if m.role == 'user' else 'Asistente'}: {m.parts[0].text}"
        for m in messages
    ])
 
    response = client.models.generate_content(
        model=MODELO,
        contents=f"Resume esta conversacion de forma concisa:\n\n{historial_texto}"
    )
 
    resumen = response.text
 
    # Reemplazamos el historial con el resumen como contexto
    return [
        types.Content(
            role="user",
            parts=[types.Part(text=f"Resumen de conversacion previa: {resumen}")]
        )
    ]
 
# Cuando usar:
# if len(streaming_conversation) > 20:
#     streaming_conversation = summarize_conversation(streaming_conversation)
 

## 11. Comparativa: Gemini vs Opciones Anteriores
 
| Caracteristica | HF Local | HF Remoto | Gemini 2.5 Flash |
|----------------|----------|-----------|------------------|
| Costo | Gratis | Free tier | Free tier (500 req/día) |
| Calidad | Media (1.5B) | Alta (72B) | Muy alta |
| Velocidad | Lenta (CPU) | Media | Muy rápida |
| Ventana de contexto | 32K tokens | 128K tokens | 1M tokens |
| Privacidad | Total | Datos en HF | Datos en Google |
| Multimodal | No | Parcial | Si (texto, imagen, audio, video) |
| Open-source | Si | Si | No |
 
## 12. Recursos para Seguir Aprendiendo
 
- [Google AI Studio](https://aistudio.google.com) — playground y gestión de keys
- [Documentación de Gemini API](https://ai.google.dev/gemini-api/docs)
- [SDK google-genai](https://googleapis.github.io/python-genai/)
- [Precios y límites](https://ai.google.dev/pricing)